# M3 (hierarchical planner) on Colab GPU

Runs Model M3 with Ollama `qwen3.5:9b` (temperature 0.6, max tokens 8192, seed 42) and writes one
Output-contract JSON per task to Google Drive.

**Resumable:** re-running a cell skips tasks whose result file already exists. If the session dropped,
just run all cells again. `--rerun-errors` also retries tasks saved as `ERROR` (e.g. Ollama died).

Runtime: *Runtime → Change runtime type → A100 GPU*.

In [ ]:
# ---- configuration ----
REPO_URL = "https://github.com/imsc-chai/298A-Team-1.git"
BRANCH = "kru/model-3"
RUN = 0                                   # -> run_00
DRIVE_ROOT = "/content/drive/MyDrive/298A"
RESULTS_DIR = f"{DRIVE_ROOT}/m3_results"  # contains run_XX/<persona>/<task>.json
CACHE_DIR = f"{DRIVE_ROOT}/model_a_cache" # OCR + embedding cache survives disconnects
REPO_DIR = "/content/298A-Team-1"
SMOKE_PERSONA, SMOKE_TASK = "persona_01_maya", "eval_03_medical_privacy_boundary"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# zstd: required by the Ollama installer to unpack its archive
!apt-get -qq update && apt-get -qq install -y tesseract-ocr zstd > /dev/null && tesseract --version | head -1
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
# Install and start Ollama in the background (log: /content/ollama.log)
!command -v zstd >/dev/null || apt-get -qq install -y zstd
!command -v ollama >/dev/null || (curl -fsSL https://ollama.com/install.sh | sh)
import subprocess, time, urllib.request
def ollama_up():
    try:
        return urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).status == 200
    except Exception:
        return False
if not ollama_up():
    subprocess.Popen("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
print("ollama up:", ollama_up())

In [ ]:
!ollama pull qwen3.5:9b
!ollama list

In [ ]:
import os
os.environ["DATA_ROOT"] = REPO_DIR            # dataset is committed at the repo root
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
os.environ["MODEL_A_CACHE"] = CACHE_DIR
os.chdir(REPO_DIR)

## 1. Smoke test: one task (Maya `eval_03_medical_privacy_boundary`)

In [ ]:
!python -m model3.run --personas {SMOKE_PERSONA} --tasks {SMOKE_TASK} --run {RUN} --results-dir {RESULTS_DIR} --rerun-errors

In [ ]:
import json
path = f"{RESULTS_DIR}/run_{RUN:02d}/{SMOKE_PERSONA}/{SMOKE_TASK}.json"
result = json.load(open(path))
view = {k: v for k, v in result.items() if k != "debug"}   # contract fields + final_response
print(json.dumps(view, indent=2))
print("\nsubgoal trace:", json.dumps(result.get("debug", {}).get("subgoal_trace"), indent=2)[:3000])

## 2. All dev tasks (maya + elena), resumable

In [ ]:
!python -m model3.run --split dev --run {RUN} --results-dir {RESULTS_DIR} --rerun-errors

## 3. Validate against the Output contract

In [ ]:
!python model3/validate_output.py {RESULTS_DIR}/run_{RUN:02d} --split dev --require-all

### Troubleshooting
- Runner says `ABORT: Ollama ...`: re-run the *start Ollama* cell, then the run cell (finished tasks are skipped).
- Ollama log: `!tail -50 /content/ollama.log`

In [ ]:
!tail -20 /content/ollama.log